# Train AI gợi ý thực tập IT — mô phỏng

Notebook huấn luyện cây quyết định độ sâu 4 trên **dữ liệu sinh viên và nhãn giả lập**. Chỉ dùng để minh họa cách làm và tạo số liệu trình bày phương pháp; không diễn giải thành hiệu quả trên sinh viên thật. Tỷ lệ chia theo hồ sơ: train 70%, validation 15%, test 15%.

Upload 3 CSV: danh sách việc làm đã gộp, hồ sơ giả lập, và cặp huấn luyện giả lập. Nhãn 0–3 được tạo bằng luật trùng kỹ năng/nhóm vị trí.


In [ ]:
from google.colab import files
import io, pandas as pd, numpy as np
up=files.upload()
def load(name):
    hits=[n for n in up if name.lower() in n.lower()]
    if len(hits)!=1: raise ValueError(f"Upload đúng một file chứa '{name}'. Đã có: {list(up)}")
    return pd.read_csv(io.BytesIO(up[hits[0]]),encoding='utf-8-sig')
jobs=load('bo_du_lieu_thuc_tap_IT')
profiles=load('mau_ho_so_sinh_vien_gia_lap')
pairs=load('mau_nhan_cap_sinh_vien_vi_tri_gia_lap')
print('Tin việc làm:',len(jobs),'| Hồ sơ giả lập:',len(profiles),'| Cặp gán nhãn:',len(pairs))
print('Nhãn:',pairs.nhan.value_counts().sort_index().to_dict())
print('Tỷ lệ hồ sơ:',profiles.split.value_counts().to_dict() if 'split' in profiles else 'xem cột split trong tập cặp')


In [ ]:
features=['tfidf_cosine','khop_dia_diem','diem_trung_binh_gia_lap']
train=pairs[pairs.split=='train']; val=pairs[pairs.split=='val']; test=pairs[pairs.split=='test']
class Node:
    def __init__(self,pred,counts,feat=None,thr=None,left=None,right=None): self.pred=pred; self.counts=counts; self.feat=feat; self.thr=thr; self.left=left; self.right=right
def gini(y):
    if len(y)==0: return 0.0
    _,c=np.unique(y,return_counts=True); p=c/len(y); return 1-float((p*p).sum())
def grow(X,y,depth=0,max_depth=4,min_leaf=8):
    vals,counts=np.unique(y,return_counts=True); node=Node(int(vals[np.argmax(counts)]),dict(zip(vals.tolist(),counts.tolist())))
    if depth>=max_depth or len(vals)==1 or len(y)<2*min_leaf: return node
    best=(0,None,None)
    for f in range(X.shape[1]):
        u=np.unique(X[:,f])
        if len(u)<2: continue
        cuts=(u[:-1]+u[1:])/2
        if len(cuts)>100: cuts=np.quantile(X[:,f],np.linspace(.05,.95,30))
        for cut in cuts:
            m=X[:,f]<=cut
            if m.sum()<min_leaf or (~m).sum()<min_leaf: continue
            gain=gini(y)-(m.sum()*gini(y[m])+(~m).sum()*gini(y[~m]))/len(y)
            if gain>best[0]: best=(gain,f,float(cut)); bestmask=m
    if best[1] is None: return node
    node.feat=best[1]; node.thr=best[2]; node.left=grow(X[bestmask],y[bestmask],depth+1,max_depth,min_leaf); node.right=grow(X[~bestmask],y[~bestmask],depth+1,max_depth,min_leaf)
    return node
def leaf(node,x):
    while node.feat is not None: node=node.left if x[node.feat]<=node.thr else node.right
    return node
class CART:
    def fit(self,df,y): self.classes_=np.arange(4); self.tree=grow(df[features].to_numpy(float),np.asarray(y,dtype=int)); return self
    def predict_proba(self,df):
        out=[]
        for x in df[features].to_numpy(float):
            n=leaf(self.tree,x); total=sum(n.counts.values()) or 1; out.append([n.counts.get(c,0)/total for c in self.classes_])
        return np.asarray(out)
    def predict(self,df): return self.classes_[self.predict_proba(df).argmax(axis=1)]
model=CART().fit(train,train.nhan)
def accuracy(y,p): return float(np.mean(np.asarray(y)==np.asarray(p)))
def macro_f1(y,p):
    vals=[]
    for c in range(4):
        tp=np.sum((np.asarray(y)==c)&(np.asarray(p)==c)); fp=np.sum((np.asarray(y)!=c)&(np.asarray(p)==c)); fn=np.sum((np.asarray(y)==c)&(np.asarray(p)!=c))
        vals.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
    return float(np.mean(vals))
if len(val): print('Validation macro-F1:',round(macro_f1(val.nhan,model.predict(val)),4))
pred=model.predict(test)
print('Test accuracy:',round(accuracy(test.nhan,pred),4))
print('Test macro-F1:',round(macro_f1(test.nhan,pred),4))
print('Test MAE:',round(float(np.mean(np.abs(test.nhan.to_numpy()-pred))),4))
print('Confusion matrix (rows true 0–3, columns predicted 0–3):\n',np.array([[int(np.sum((test.nhan.to_numpy()==a)&(pred==b))) for b in range(4)] for a in range(4)]))


In [ ]:
# Xếp hạng Top-K theo giá trị kỳ vọng của nhãn dự đoán (0–3), không phải xác suất được nhận.
test=test.copy()
proba=model.predict_proba(test[features])
test['score']=proba @ model.classes_
test['relevant']=(test.nhan>=2).astype(int)
def ranking_metrics(g,k):
    g=g.sort_values('score',ascending=False); rel=g.nhan.to_numpy()[:k]
    dcg=sum((2**r-1)/np.log2(i+2) for i,r in enumerate(rel))
    ideal=np.sort(g.nhan.to_numpy())[::-1][:k]
    idcg=sum((2**r-1)/np.log2(i+2) for i,r in enumerate(ideal))
    hits=int((g.relevant.to_numpy()[:k]).sum()); total=int(g.relevant.sum())
    return (dcg/idcg if idcg else np.nan, hits/min(k,len(g)), hits/total if total else np.nan)
for k in (5,10):
    vals=[ranking_metrics(g,k) for _,g in test.groupby('ma_ho_so')]
    print(f'NDCG@{k}:',round(float(np.nanmean([x[0] for x in vals])),4), '| Precision@K:',round(float(np.nanmean([x[1] for x in vals])),4),'| Recall@K:',round(float(np.nanmean([x[2] for x in vals])),4))


In [ ]:
test['du_doan_nhan']=model.predict(test[features])
meta=jobs[['ma_jd','job_title','company_name','location','source_dataset']].drop_duplicates('ma_jd')
test=test.merge(meta,on='ma_jd',how='left')
test.to_csv('ket_qua_test_gia_lap_colab.csv',index=False,encoding='utf-8-sig')
top=(test.sort_values(['ma_ho_so','score'],ascending=[True,False]).groupby('ma_ho_so').head(10).copy())
top['thu_hang']=top.groupby('ma_ho_so')['score'].rank(method='first',ascending=False).astype(int)
top.to_csv('top_k_test_gia_lap_colab.csv',index=False,encoding='utf-8-sig')
files.download('ket_qua_test_gia_lap_colab.csv')
files.download('top_k_test_gia_lap_colab.csv')


## Lưu ý báo cáo
Các điểm số sẽ phản ánh mức độ mô hình khôi phục nhãn tổng hợp theo luật đã định nghĩa. Dữ liệu chưa có phản hồi thực tế của sinh viên/giảng viên; không dùng kết quả mô phỏng như độ chính xác của hệ thống triển khai. Khi có nhãn thật, thay hai tệp giả lập bằng cặp đã đánh giá và chạy lại toàn bộ.
